# Dataset-usage detection on Colab (32 px)

**Before running:** Runtime → Change runtime type → **T4 GPU**. In the key icon on the left (Secrets), add `SHIELD_KEY` with a long random value and allow this notebook to read it. Use the same value in every session: the owner's message is derived from it, so a different key means a different message, and earlier results stop matching.

Everything is saved to your Google Drive, so finished work is skipped when you run a cell again.

**After any runtime restart, run cells 1–3 again** before anything else.

In [ ]:
# 1. Is there a GPU?
!nvidia-smi -L || echo 'NO GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Keep results on Drive, and read the secret key
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['SHIELD_WORK'] = '/content/drive/MyDrive/kondor-shield/work'
os.environ['SHIELD_OUT'] = '/content/drive/MyDrive/kondor-shield/out'
os.environ['SHIELD_KEY'] = userdata.get('SHIELD_KEY')

In [ ]:
# 3. Get the code and install
%cd /content
!test -d kondor || git clone --branch dataset-usage-detection --single-branch https://github.com/0vertake/kondor.git
%cd /content/kondor
!git pull -q
!pip uninstall -q -y torchao
!pip install -q -r requirements.txt
!python -m pytest -q tests

## Stage 1 and E2: train the engine, then test transfer at 100% marked
Skipped if already done. Takes about 20 minutes from scratch.

In [ ]:
%cd /content/kondor
!python stage1_engine.py --config px32
!python stage5_transfer.py --steps 2000 --n 500 --sample-batch 128

## Step 1: why did the clean model score 61.7%?
Scores the saved images against 2,000 other owners' messages. Trains nothing; takes minutes. Look for the `FALSELY DETECTED` counts, and for how many other owners score as high as the real one.

In [ ]:
%cd /content/kondor
!python exp_owner_scan.py --owners 2000

## Step 2 (E3): how much of a dataset must be marked?
Fine-tunes one model per marked share, about 8 minutes each. Reuses the clean and 100% models. If the session drops, run it again: finished shares are skipped.

In [ ]:
%cd /content/kondor
!python stage5_transfer.py --steps 2000 --n 500 --sample-batch 128 --shares 1.0 0.5 0.2 0.05 0.01

## Look at the results
The curve, then images from the clean model and the 100% marked model. Marking should not visibly hurt them.

In [ ]:
from IPython.display import Image, display
run = '/content/drive/MyDrive/kondor-shield/out/e2/px32_s2000_b32_lr0.0001/'
for name in ['e3_curve.png', 'clean_samples.png', 'marked1_samples.png']:
    print(name)
    display(Image(run + name))